<a href="https://colab.research.google.com/github/italobotelho/PI2026.1/blob/main/01_ETL_Extracao_SINAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install PySUS

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import os
from pysus.ftp.databases.sinan import SINAN

# ---------------------------------------------------------
# 1. CONFIGURAÇÕES GERAIS DO PROJETO SIEST
# ---------------------------------------------------------

# Inicializa o módulo do SINAN
sinan = SINAN().load()

# Código IBGE de Campinas
IBGE_CAMPINAS = 350950

# Janela temporal: 2014 a 2026 (o range no Python vai até o número anterior ao limite)
anos_analise = range(2014, 2027)

# Lista de agravos (Doenças) baseada no ODS 13 e integração epidemiológica
# DENG (Dengue), CHIK (Chikungunya), ZIKA (Zika), LEPT (Leptospirose), HEPA (Hepatites Virais)
doencas = ['DENG', 'CHIK', 'ZIKA', 'LEPT', 'HEPA']

# Lista expandida das colunas de interesse (O "Dicionário" do SIEST)
colunas_alvo = [
    'ID_MUNICIP',  # Para garantir o filtro espacial
    'ID_UNIDADE',  # CNES: O Nó do nosso Grafo
    'DT_NOTIFIC',  # Data da notificação
    'DT_SIN_PRI',  # Data dos primeiros sintomas (Crucial para clima)
    'NU_IDADE_N',  # Idade do paciente
    'CS_SEXO',     # Sexo
    'HOSPITALIZ',  # Indicador de sobrecarga hospitalar (1-Sim, 2-Não)
    'CLASSI_FIN',  # Classificação final do caso
    'EVOLUCAO',    # Evolução (Cura, Óbito, etc)
    'CRITERIO'     # Critério de confirmação
]

# Cria um diretório para organizar os dados limpos
os.makedirs('dados_siest_parquet', exist_ok=True)

# ---------------------------------------------------------
# 2. MOTOR DE EXTRAÇÃO E TRANSFORMAÇÃO (ETL)
# ---------------------------------------------------------

print("Iniciando o Pipeline de ETL do SIEST...")

# Loop Duplo: Para cada doença, varre todos os anos
for doenca in doencas:
    print(f"\\n--- Extraindo dados para: {doenca} ---")

    for ano in anos_analise:
        try:
            # Extração dos arquivos do FTP do DataSUS
            arquivos = sinan.get_files(doenca, ano)

            # Se não houver arquivos para a doença/ano, pula para o próximo
            if not arquivos:
                print(f"[{ano}] Sem arquivos no servidor para {doenca}.")
                continue

            df_bruto = sinan.download(arquivos).to_dataframe()

            # Transformação 1: Converter ID_MUNICIP para numérico para aplicar o filtro
            df_bruto['ID_MUNICIP'] = pd.to_numeric(df_bruto['ID_MUNICIP'], errors='coerce')

            # Transformação 2: Filtro Espacial (Apenas Campinas)
            df_campinas = df_bruto[df_bruto['ID_MUNICIP'] == IBGE_CAMPINAS].copy()

            # Se não houver casos em Campinas neste ano, pula
            if df_campinas.empty:
                print(f"[{ano}] 0 casos registrados em Campinas para {doenca}.")
                continue

            # Transformação 3: Interseção de Colunas (Evita o KeyError)
            # Pega apenas as colunas da nossa lista que realmente existem no arquivo atual
            colunas_existentes = df_campinas.columns.intersection(colunas_alvo)
            df_campinas = df_campinas[colunas_existentes]

            # Carga (Load): Salvar em Parquet com compressão
            nome_arquivo = f"dados_siest_parquet/{doenca}_campinas_{ano}.parquet"
            df_campinas.to_parquet(nome_arquivo, index=False, compression='snappy')

            print(f"[{ano}] Sucesso! {len(df_campinas)} casos de {doenca} salvos.")

            # Limpeza de memória RAM no Colab
            del df_bruto
            del df_campinas

        except Exception as e:
            print(f"[{ano}] Erro ao processar {doenca}: {e}")

print("\\nPipeline concluído! Todos os arquivos limpos e comprimidos foram salvos.")

Iniciando o Pipeline de ETL do SIEST...
\n--- Extraindo dados para: DENG ---


DENGBR14.parquet: 100%|██████████| 1.55M/1.55M [02:39<00:00, 9.72kB/s]


[2014] Sucesso! 47646 casos de DENG salvos.


DENGBR15.parquet: 100%|██████████| 3.85M/3.85M [06:38<00:00, 9.65kB/s]


[2015] Sucesso! 77190 casos de DENG salvos.


...

64655421it [00:00, 36259458607.03it/s]


[2022] Sucesso! 12446 casos de DENG salvos.


DENGBR23.parquet: 100%|██████████| 2.68M/2.68M [04:36<00:00, 9.69kB/s]


[2023] Sucesso! 12078 casos de DENG salvos.


DENGBR24.parquet: 100%|██████████| 10.7M/10.7M [18:51<00:00, 9.46kB/s]


In [ ]:
#import pandas as pd
#import os
import shutil
#from pysus.ftp.databases.sinan import SINAN

# ---------------------------------------------------------
# 1. CONFIGURAÇÕES GERAIS DO PROJETO SIEST
# ---------------------------------------------------------
sinan = SINAN().load()

IBGE_CAMPINAS = 350950
anos_analise = range(2014, 2027)
doencas = ['DENG', 'CHIK', 'ZIKA', 'LEPT', 'HEPA']

colunas_alvo = [
    'ID_MUNICIP', 'ID_UNIDADE', 'DT_NOTIFIC', 'DT_SIN_PRI',
    'NU_IDADE_N', 'CS_SEXO', 'HOSPITALIZ', 'CLASSI_FIN', 'EVOLUCAO', 'CRITERIO'
]

os.makedirs('dados_siest_parquet', exist_ok=True)

# ---------------------------------------------------------
# 2. MOTOR DE ETL COM PROCESSAMENTO EM LOTES (CHUNKING)
# ---------------------------------------------------------

print("Iniciando ETL com Processamento em Lotes (Anti-OOM)...")

for doenca in doencas:
    print(f"\n--- Extraindo dados para: {doenca} ---")

    for ano in anos_analise:
        try:
            arquivos = sinan.get_files(doenca, ano)

            if not arquivos:
                print(f"[{ano}] Sem arquivos no servidor para {doenca}.")
                continue

            # Faz o download para o cache, mas NÃO usa o .to_dataframe()!
            # Isso retorna o caminho da pasta onde o PySUS salvou os pequenos arquivos
            pasta_cache = sinan.download(arquivos)
            caminho_pasta = str(pasta_cache)

            # Lista todos os 'pedaços' (chunks) baixados
            chunks = [f for f in os.listdir(caminho_pasta) if f.endswith('.parquet')]
            lista_campinas = []

            # Lê, filtra e DESCARTA arquivo por arquivo!
            for chunk in chunks:
                caminho_arquivo = os.path.join(caminho_pasta, chunk)

                # Lê apenas um pequeno pedaço da base
                df_chunk = pd.read_parquet(caminho_arquivo)

                # Aplica o filtro de Campinas IMEDIATAMENTE
                df_chunk['ID_MUNICIP'] = pd.to_numeric(df_chunk['ID_MUNICIP'], errors='coerce')
                df_filtrado = df_chunk[df_chunk['ID_MUNICIP'] == IBGE_CAMPINAS].copy()

                if not df_filtrado.empty:
                    # Mantém apenas as colunas alvo
                    colunas_existentes = df_filtrado.columns.intersection(colunas_alvo)
                    lista_campinas.append(df_filtrado[colunas_existentes])

                # DELETA o chunk bruto gigante da memória RAM
                del df_chunk
                del df_filtrado

            # Se encontrou casos em Campinas ao longo dos chunks
            if lista_campinas:
                # Junta apenas os casos de Campinas (que cabem folgadamente na RAM)
                df_campinas_final = pd.concat(lista_campinas, ignore_index=True)

                nome_arquivo = f"dados_siest_parquet/{doenca}_campinas_{ano}.parquet"
                df_campinas_final.to_parquet(nome_arquivo, index=False, compression='snappy')

                print(f"[{ano}] Sucesso! {len(df_campinas_final)} casos salvos.")
                del df_campinas_final
            else:
                print(f"[{ano}] 0 casos registrados em Campinas para {doenca}.")

            # Limpa o HD apagando a pasta de cache do PySUS daquele ano
            shutil.rmtree(caminho_pasta, ignore_errors=True)

        except Exception as e:
            print(f"[{ano}] Erro ao processar {doenca}: {e}")

print("\nPipeline concluído com sucesso!")

Iniciando ETL com Processamento em Lotes (Anti-OOM)...

--- Extraindo dados para: DENG ---


42676892it [00:00, 6510032689.23it/s]


[2014] Sucesso! 47646 casos salvos.


109231410it [00:00, 15981224357.77it/s]


[2015] Sucesso! 77190 casos salvos.


...

[2021] Sucesso! 270 casos salvos.


HEPABR22.parquet: 100%|██████████| 35.8k/35.8k [00:04<00:00, 7.86kB/s]


[2022] Sucesso! 265 casos salvos.


HEPABR23.parquet: 100%|██████████| 38.9k/38.9k [00:03<00:00, 10.9kB/s]

[2023] Sucesso! 417 casos salvos.
[2024] Sem arquivos no servidor para HEPA.
[2025] Sem arquivos no servidor para HEPA.
[2026] Sem arquivos no servidor para HEPA.

Pipeline concluído com sucesso!
